# Sensor Coregistration

For proper computation, the sensors (electrode channels or magnetic coils) must be aligned to the head model. <br>
Alignment steps:
1. Visualize initial alignment
2. Align fiducials (Aligns reference points, serves to align roughly and fast before refining)
3. Refine alignment using an ICP algorithm
4. Project points to head (🚨 I would only use this step if the shape of the EEG cap does not match the shape of the head. This can happend when either the EEG montage or Head model was taken from a template. I do this in simulations where the true location of electrodes is arbitrary anyways 🚨)
5. Save the transformation file. (Save even if you don't apply transformations. This will save an identity transformation.)

## 📦 Imports

In [1]:
import mindmap as mm 
import mne
import pyvista as pv

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
**Anatomy**
- `bem`: Boundary Element Model (head model)

**Functional**
- `eeg-raw`: `mne.io.Raw`, `mne.Epochs` or `mne.Evoked` EEG data.

🎚️ **<font color=pink>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)

In [2]:
# ------------ Parameters -------------
fconfig = "../../papers/simf/configuration.yml"
sub     = 1
# -------------------------------------

reader = mm.io.dataset_reader(fconfig)
fs_dir = reader.config["FS_DIR"]
mm_dir = reader.config["MM_DIR"]

data = reader(sub, "eeg-raw", "bem")


DATA SUMMARY
Dataset: simf
Subject: vepcon_sub-10
Retrieved objects:
- eeg-raw
- bem
Missing objects:


## ⚙️ Coregister sensors
📊 **Current alignment**: after each registration step, come back to this plot to see the changes

In [3]:
# outer skin surface
head = data["bem"]["surfs"][0]
head = mm.head.surf_to_polydata(head["rr"], head["tris"])

# coregistration object (computes MRI fiducials from fsaverage)        
coreg   = mne.coreg.Coregistration(data["eeg-raw"].info, data["sub"], fs_dir, fiducials="estimated")
fid_mri = pv.PolyData([coreg.fiducials.get_positions()[p] for p in ["nasion", "lpa", "rpa"]])

# sensor positions
sens      = pv.PolyData(data["eeg-raw"]._get_channel_positions())
fid_sens  = pv.PolyData(mm.sensor.get_fiducials(data["eeg-raw"]))

# plot
plotter = pv.Plotter()
plotter.add_mesh(head, label="head", smooth_shading=False, color="#D2A18C")
plotter.add_mesh(fid_mri, render_points_as_spheres=True, color="red", label="MRI fiducials", point_size=10)
plotter.add_mesh(sens, render_points_as_spheres=True, color="black", label="sensors", point_size=8)
plotter.add_mesh(fid_sens, render_points_as_spheres=True, color="purple", label="sensor fiducials", point_size=10)

plotter.add_legend(face="circle")
plotter.show()

    Triangle neighbors and vertex normals...
Using low resolution head model in /media/nino/DATA/SIMF/derivatives/freesurfer/vepcon_sub-10/bem/outer_skin.surf
    Triangle neighbors and vertex normals...
Estimating fiducials from fsaverage.


Widget(value='<iframe src="http://localhost:40963/index.html?ui=P_0x71f6f2484550_0&reconnect=auto" class="pyvi…

#### Fit fiducials

Changes are updated in the visualization above.

In [4]:
# fit fiducials
coreg.fit_fiducials(verbose="warning")

# plot
sens.points     = mne.transforms.apply_trans(coreg.trans, data["eeg-raw"]._get_channel_positions())
fid_sens.points = mne.transforms.apply_trans(coreg.trans, mm.sensor.get_fiducials(data["eeg-raw"]))

#### Apply Iteratice Closest Point (ICP)
Changes are updated in the visualization above.

🎚️ **<font color=pink>ICP</font>**:
- `iterations`: Number of ICP iterations.
- `nasion_weight`: ???

In [5]:
# ------------ Parameters -------------
iterations    = 30
nasion_weight = 1
# -------------------------------------

# icp
coreg.fit_icp(n_iterations=iterations, nasion_weight=nasion_weight, verbose="warning")

# plot
sens.points     = mne.transforms.apply_trans(coreg.trans, data["eeg-raw"]._get_channel_positions())
fid_sens.points = mne.transforms.apply_trans(coreg.trans, mm.sensor.get_fiducials(data["eeg-raw"]))

#### Project to surface
Only recommended for fitting template sensors to MRI in simulations. The head model has low resolution so the vertices are quite far from each other for accurately projecting to the head.


In [7]:
if data["name"] == "simf":
    points = mm.head.project_points_to_surface(mne.transforms.apply_trans(coreg.trans, data["eeg-raw"]._get_channel_positions()), head)
    sens.points = points

    # create eeg with new points
    channels = dict(zip(data["eeg-raw"].info["ch_names"], points))
    montage  = mne.channels.make_dig_montage(ch_pos=channels, coord_frame="head")
    eeg      = data["eeg-raw"].set_montage(montage)

💾 **Save trans file** <br>
If you don't apply any transformation, still save the `coreg.trans` file — it will contain an identity transformation. Ideally, you should always save the transformation file. For the simf data, we project the sensors onto the surface so that they fit the head shape. Because of this projection, the transformation is no longer linear and cannot be used by MNE. Instead, save the "eeg-coreg" file, which includes the correct sensor placement.

🎚️ **<font color=pink>save</font>**:
- `overwrite`: Overwrite existing files if True

In [8]:
# ------------ Parameters -------------
overwrite = True
# -------------------------------------

if data["name"] == "simf":
    # save coregistered EEG
    feeg = reader.config["MM_DIR"]/data["sub"]/"eeg"/f"{data['sub']}_desc-empty-coreg_ave.fif"

    if not feeg.parent.exists():
        feeg.parent.mkdir(parents=True)
    
    eeg.save(feeg, overwrite=overwrite, verbose="warning")
    print(f"\033[92mSuccessfully saved EEG with new channel positions for subject {data['sub']}\033[0m")

    # Reinitialize coregistration to save identity matrix as transformation.
    coreg = mne.coreg.Coregistration(data["eeg-raw"].info, data["sub"], fs_dir, fiducials="estimated")

# save transformation
ftrans = reader.config["MM_DIR"]/data["sub"]/"eeg"/f"{data['sub']}_trans.fif"

if not ftrans.parent.exists():
    ftrans.parent.mkdir(parents=True)

mne.write_trans(ftrans, coreg.trans, overwrite=overwrite)
print(f"\033[92mSuccessfully saved EEG transformation for subject {data['sub']}\033[0m")

Successfully saved EEG with new channel positions for subject vepcon_sub-10
    Triangle neighbors and vertex normals...
Using low resolution head model in /media/nino/DATA/SIMF/derivatives/freesurfer/vepcon_sub-10/bem/outer_skin.surf
    Triangle neighbors and vertex normals...
Estimating fiducials from fsaverage.
Successfully saved EEG transformation for subject vepcon_sub-10
